# 같은 PDF, 파서마다 다른 결과 (2회차 노트북)

**같은 PDF 인데 파서에 따라 결과가 다르다.** 이 노트북은 그것을 눈으로 보는 것이 목적이다.
결론을 먼저 말하면 이렇다.

> 파서를 하나만 고르지 않는다. **기본 파서로 뽑고, 품질을 숫자로 재서, 나쁘면 다른 파서로 다시 뽑는다.**

이 노트북에서 재는 숫자가 그대로 `validate.py` 의 기준값이 된다.
오늘 그 함수를 여러분이 직접 채운다.

셀 번호는 코드 셀 첫 줄의 `# [셀 N]` 표시다. 커널을 재시작하고 Run All 하면 왼쪽의 `[N]` 도 같은 번호가 된다.

**필요한 것**: 0·1회차에서 받은 코퍼스. LLM 키는 필요 없다.

In [ ]:
# [셀 1] 준비 — 파서 세 개와 문서 목록 불러오기
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import csv
from finrag.parsing.extract import _pdfplumber, _pymupdf, _pypdf, HANGUL, broken_ratio
from finrag.parsing.validate import score_page

PARSERS = {"pdfplumber": _pdfplumber, "pymupdf": _pymupdf, "pypdf": _pypdf}

MANIFEST = {r["doc_id"]: r for r in csv.DictReader((ROOT / "data/corpus_manifest.csv").open(encoding="utf-8"))}

def path_of(doc_id: str) -> Path:
    """매니페스트에서 파일 위치를 찾는다. bundled 는 data/bundled, 나머지는 data/raw."""
    r = MANIFEST[doc_id]
    base = "data/bundled" if r["tier"] == "bundled" else "data/raw"
    return ROOT / base / r["filename"]

print("코퍼스", len(MANIFEST), "건 ·", sum(1 for d in MANIFEST if path_of(d).exists()), "건 확보")

## 1부 · 같은 PDF, 세 파서

먼저 비교 함수를 만든다. 재는 것은 다섯 가지다.

| 지표 | 무엇을 보려고 재는가 |
|---|---|
| 글자 수 | 아무것도 못 뽑았는지 |
| **한글 수** | 뽑긴 뽑았는데 한국어가 아닌지 |
| 공백 비율 | 띄어쓰기가 사라졌는지 |
| 깨진 문자 비율 | 폰트 매핑이 깨졌는지 (PUA · `(cid:123)`) |
| 단음절 토큰 비율 | 한 글자짜리 토막이 얼마나 많은지 (`금 융 소 비 자`) |

In [ ]:
# [셀 2] 비교 함수 compare · show, 하나은행 2009 여신거래기본약관
import time

def compare(doc_id: str, parsers=PARSERS) -> dict:
    """세 파서를 같은 문서에 돌려 지표를 나란히 놓는다."""
    p = path_of(doc_id)
    out = {}
    for name, fn in parsers.items():
        t0 = time.perf_counter()
        try:
            pages = fn(p)
        except Exception as e:
            out[name] = {"error": f"{type(e).__name__}"}
            continue
        text = "\n".join(pages)
        m = score_page(text)
        out[name] = {"쪽": len(pages), "글자": len(text), "한글": len(HANGUL.findall(text)),
                     "공백": m["space_ratio"], "깨짐": m["broken_ratio"],
                     "단음절": m["single_char_token_ratio"], "초": round(time.perf_counter() - t0, 2)}
    return out

def show(doc_id: str):
    print(f"{doc_id}   ({MANIFEST[doc_id]['title'][:40]})")
    print(f"  {'파서':12}{'글자':>9}{'한글':>9}{'공백':>8}{'깨짐':>9}{'단음절':>8}{'초':>7}")
    for name, r in compare(doc_id).items():
        if "error" in r:
            print(f"  {name:12} 실패 {r['error']}"); continue
        print(f"  {name:12}{r['글자']:>9,}{r['한글']:>9,}{r['공백']:>8.4f}"
              f"{r['깨짐']:>9.5f}{r['단음절']:>8.3f}{r['초']:>7}")

show("hana_credit_terms_2009")

### 함정 하나

**글자 수만 보면 pdfplumber 가 가장 많다.** 76,979자 대 7,847자. 열 배 가까이 된다.

그런데 **한글이 0자다.**

pdfplumber 가 뽑은 76,979자는 전부 한국어가 아닌 무언가다. 이 문서는 2009년에
만들어진 PDF 고, 폰트에 "이 번호는 이 글자다"라는 표(ToUnicode)가 제대로 안 들어 있다.
pdfplumber 는 그 표가 없으면 글자 번호를 `(cid:1104)` 처럼 그대로 문자로 내놓는다.

글자 수가 열 배인 이유도 이것이다. 글자 하나가 `(cid:1104)` 라는 열 글자짜리 번호로 나오니
글자 수만 열 배로 부푼다. 76,979자 중 97% 가 이 번호다.

**"글자가 나왔다"와 "읽을 수 있다"는 다르다.** 실제로 보자.

In [ ]:
# [셀 3] 하나은행 2009 — 세 파서가 뽑은 글자의 앞부분
p = path_of("hana_credit_terms_2009")
for name, fn in PARSERS.items():
    text = "\n".join(fn(p))
    body = text.strip().replace("\n", " ")
    print(f"[{name}]")
    print("   ", repr(body[:110]))
    print()

세 개가 같은 페이지다.

- `pdfplumber` — 글자 번호와 기호가 섞인 무더기. 사람이 못 읽는다.
- `pymupdf` — 정상. 이 문서에서는 이것만 쓸 수 있다.
- `pypdf` — 772자에 깨짐 비율 0.67. 거의 전부 깨졌다.

> **왜 pdfplumber 를 기본으로 두는가?**
> 이 문서만 보면 PyMuPDF 가 더 낫지만, 다음 문서에서는 반대가 된다.

In [ ]:
# [셀 4] 금감원 예금거래기본약관 — 세 파서 비교
show("fss_deposit_terms_2024_pdf")

### 결함 2 — 띄어쓰기가 사라진다

한글 수는 셋 다 **5,625자로 같다.** 글자는 다 뽑았다. 그런데 공백 비율이 다르다.

- pdfplumber `0.206`
- **pymupdf `0.026`** ← 8배 적다
- pypdf `0.195`

PyMuPDF 가 띄어쓰기를 잃었다. 무슨 뜻인지 눈으로 보자.

In [ ]:
# [셀 5] 금감원 예금거래기본약관 — 띄어쓰기 비교
p = path_of("fss_deposit_terms_2024_pdf")
for name in ("pdfplumber", "pymupdf"):
    text = "\n".join(PARSERS[name](p))
    # 같은 대목을 찾아 비교한다
    i = text.find("예금거래")
    print(f"[{name}]  공백비율 {score_page(text)['space_ratio']:.4f}")
    print("   ", repr(text[i:i+90]) if i >= 0 else repr(text[:90]))
    print()

띄어쓰기가 없으면 무엇이 깨지는가.

1. **키워드 검색이 단어를 못 나눈다** → BM25 색인이 망가진다 (4회차)
2. **사람이 근거를 읽을 수 없다** → 인용을 붙여도 못 읽는다
3. 조항 제목을 정규식으로 못 찾는다 → 청킹이 어긋난다 (오늘 실습 3)

그래서 이 문서는 **pdfplumber 가 맞다.** 앞 문서는 PyMuPDF 가 맞았다.
**문서마다 답이 다르다.**

In [ ]:
# [셀 6] 손보협회 스캔본 — 세 파서 비교
show("knia_3500_scan")

### 결함 3 — 글자 층이 아예 없다

셋 다 7자다. 파서 문제가 아니라 **PDF 안에 글자가 없다.** 스캔 이미지다.

여기서 파서를 더 바꿔 봐야 소용없다. **OCR 로 가야 한다** (3부).

판정 규칙이 하나 나온다:

> 쪽당 글자 수가 기준 미만인데 **이미지가 있으면** → OCR
> 이미지도 없으면 → 그냥 빈 페이지. 통과시킨다.

빈 페이지를 실패로 처리하면 목차·간지가 전부 검토 큐로 간다.

### 결함 4 — 한 쪽만 깨졌다

지금까지는 문서 전체가 문제였다. 이번엔 **492쪽 중 한 쪽**이다.

In [ ]:
# [셀 7] 실손 표준약관 492쪽 — 깨진 쪽 찾기
import collections

pages = _pymupdf(path_of("law_silson_std_pdf"))
scores = [(score_page(t)["broken_ratio"], i) for i, t in enumerate(pages, 1) if len(t) >= 50]
worst = sorted(scores, reverse=True)[:5]
print(f"전체 {len(pages)}쪽 중 깨짐 비율 상위 5쪽")
for r, i in worst:
    print(f"   {i:>4}쪽  {r:.5f}")

bad_page = pages[worst[0][1] - 1]
pua = collections.Counter(c for c in bad_page if 0xE000 <= ord(c) <= 0xF8FF)
print(f"\n{worst[0][1]}쪽: PUA 문자 {sum(pua.values())}개 · 종류 {len(pua)}종")
print("   ", [(hex(ord(c)), n) for c, n in pua.most_common(3)])

j = next(k for k, c in enumerate(bad_page) if 0xE000 <= ord(c) <= 0xF8FF)
print("\n문맥:")
print("   ", repr(bad_page[max(0, j - 60):j + 40]))

U+F000 은 **사설 영역(Private Use Area)** 문자다. 폰트 제작자가 마음대로
쓰라고 비워 둔 자리라 **무슨 글자인지 정해져 있지 않다.** 표 안의 괘선이나 특수 기호를
그 자리에 넣은 것으로 보인다.

문맥을 보면 하나가 더 보인다. `보험금을지급할때의적립이율계산` — **띄어쓰기도 없다.**
한 쪽에 결함이 둘이다.

> **여기서 하지 말아야 할 일: 492쪽짜리 문서를 통째로 버리는 것.**
> 실제로 그렇게 했다가 골든셋 근거 문서 5건을 잃은 적이 있다.
> 나쁜 쪽은 표시만 하고, 나머지 491쪽은 색인한다.

### 그래서 파서를 고르는 규칙

| 문서 | 증상 | 답 |
|---|---|---|
| 하나은행 2009 약관 | pdfplumber 한글 0자 | PyMuPDF |
| 금감원 예금거래약관 | PyMuPDF 공백 0.026 | pdfplumber |
| 손보협회 스캔본 | 셋 다 7자 | OCR |
| 실손 표준약관 55쪽 | PUA 89개 | 그 쪽만 표시, 나머지 색인 |

**기본은 pdfplumber, 안 되면 PyMuPDF.** 기본을 pdfplumber 로 두는 이유는
띄어쓰기를 잘 지키고, 표를 뽑을 수 있기 때문이다(2부).

그러면 "안 되면 PyMuPDF" 가 실제로 몇 번이나 일어나는가? 1회차에서 만든 프로파일로 확인한다.

In [ ]:
# [셀 8] 파서에 따라 결과가 달라지는 문서 세기
import json

profile = json.loads((ROOT / "data/doc_profile.json").read_text(encoding="utf-8"))

# (1) 파서에 따라 한글이 나오거나 안 나오는 문서
#     pypdf 와 pymupdf 만 비교한다. pdfplumber 는 느려서 프로파일러가 5쪽만 표본으로
#     재기 때문에(pdfplumber_sample) 전체 값과 나란히 놓으면 안 된다.
hangul_split = []
for did, d in profile.items():
    ex = d.get("extractors") or {}
    if not {"pypdf", "pymupdf"} <= set(ex):
        continue
    h = {k: ex[k]["hangul"] for k in ("pypdf", "pymupdf")}
    if max(h.values()) > 0 and max(h.values()) >= min(h.values()) * 2:
        hangul_split.append((did, h))

# (2) 파서에 따라 띄어쓰기가 살거나 사라지는 문서 (같은 표본 5쪽으로 잰 값끼리 비교)
space_split = [(d["space_rate_pymupdf"], d["space_rate_pdfplumber"], did)
               for did, d in profile.items()
               if d.get("space_rate_pymupdf") is not None and d.get("space_rate_pdfplumber")]
space_gap = [t for t in space_split if t[1] >= t[0] * 2]

print(f"프로파일된 문서 {len(profile)}건")
print(f"\n파서에 따라 한글이 달라지는 문서: {len(hangul_split)}건")
for did, h in hangul_split:
    print(f"   {did:32}", {k: f"{v:,}" for k, v in h.items()})
print(f"\n파서에 따라 띄어쓰기가 달라지는 문서: {len(space_gap)}건 / 비교 가능 {len(space_split)}건")
for mu, pl, did in sorted(space_gap):
    print(f"   {did:32} pymupdf {mu:.4f}   pdfplumber {pl:.4f}")

**파서에 따라 결과가 달라지는 문서는 51건 중 6건이다.** 나머지 45건은 어느 파서를 써도 같다.

- 파서에 따라 **한글이 나오거나 안 나오는** 문서 **1건** — 하나은행 2009. pypdf 로는 한글이 0자다.
- 파서에 따라 **띄어쓰기가 살거나 사라지는** 문서 **5건** — 전부 pdfplumber 쪽이 낫다.

여기서 설계 원칙이 나온다.

> **그 6건을 위해 파이프라인 전체를 복잡하게 만들지 않는다.**
> 기본 경로는 단순하게 두고, **품질을 재 봐서 기준에 못 미친 것만** 다른 길로 보낸다.

이게 오늘 만들 검증 게이트(검사기)다. 게이트가 없으면 둘 중 하나를 하게 된다 —
모든 문서에 세 파서를 다 돌리거나(느리다), 하나만 쓰고 6건을 잃거나.

> **주의**: 위 비교에서 pdfplumber 는 5쪽 표본으로만 쟀다. 프로파일러가 전체를
> 안 도는 이유는 이 노트북 1부에서 본 그대로다 — pdfplumber 는 PyMuPDF 보다
> 10배 넘게 느리다(0.5초 대 0.04초).

## 2부 · 표

약관·상품설명서는 표가 본문만큼 중요하다. 중도해지이율, 수수료율, 자기부담금이
전부 표 안에 있다. 그런데 표는 **그냥 글자로 펴 버리면 의미가 무너진다.**

In [ ]:
# [셀 9] 하나카드 개정대비표 — 표 추출과 None 칸
import pdfplumber

with pdfplumber.open(path_of("hanacard_diff_2016")) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    t = tables[0]

print(f"표 {len(tables)}개 · 첫 표 {len(t)}행 × {len(t[0])}열")
none_cells = sum(1 for row in t for c in row if c is None)
print(f"None 셀: {none_cells} / {sum(len(r) for r in t)}\n")
for row in t[:2]:
    print([(c[:16] + "…" if c and len(c) > 16 else c) for c in row])

### 합쳐진 칸은 `None` 이 된다

이 문서는 **개정대비표**다. 「현행 | 개정(안) | 비고」 3단인데, 각 단이 다시
3열을 합쳐 하나로 돼 있다. pdfplumber 는 합쳐진 칸을 이렇게 표현한다.

- 합쳐진 칸의 **첫 칸**: 실제 값
- 합쳐진 칸의 **나머지 칸**: `None`

`None` 을 그냥 버리고 이어 붙이면 `현행 개정(안) 비고` 가 되어 **어느 값이 어느
단인지 사라진다.** 개정 전후가 뒤섞이면 "무엇이 바뀌었나"를 답할 수 없다.

18칸 중 6칸이 `None` 이다. 3분의 1이다.

In [ ]:
# [셀 10] 표를 글자로 — 마크다운 표 · 행 단위 문장
def to_markdown(table) -> str:
    """표를 마크다운으로. None 은 앞 칸 값을 이어받는다(병합 복원)."""
    rows = []
    for row in table:
        filled, last = [], ""
        for c in row:
            last = (c or last).replace("\n", " ").strip()
            filled.append(last)
        rows.append(filled)
    head, *body = rows
    out = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    out += ["| " + " | ".join(r) + " |" for r in body]
    return "\n".join(out)


def to_sentences(table) -> list[str]:
    """표를 행 단위 문장으로. 헤더를 각 값에 붙여 준다."""
    rows = []
    for row in table:
        filled, last = [], ""
        for c in row:
            last = (c or last).replace("\n", " ").strip()
            filled.append(last)
        rows.append(filled)
    head, *body = rows
    out = []
    for r in body:
        parts = [f"{h}: {v}" for h, v in zip(head, r) if v and h]
        if parts:
            out.append(", ".join(parts))
    return out


print("── 마크다운 표 ──")
print(to_markdown(t)[:400])
print("\n── 행 단위 문장 ──")
for s in to_sentences(t)[:2]:
    print(" ", s[:180])

### 어느 쪽이 검색에 유리한가

| | 마크다운 표 | 행 단위 문장 |
|---|---|---|
| 사람이 읽기 | 좋다 | 답답하다 |
| 원래 구조 보존 | 좋다 | 헤더가 값마다 반복된다 |
| **임베딩 검색** | 표 전체가 한 덩어리 → 특정 행이 안 걸린다 | **행 하나가 한 청크** → 잘 걸린다 |
| **BM25(키워드 검색)** | 헤더가 한 번만 나온다 | 헤더 단어가 행마다 → 점수 왜곡 |
| 근거 인용 | 표 전체를 보여 줘야 한다 | 해당 행만 인용 |

**정답이 없다.** 4회차에 두 방식으로 색인해서 Recall 을 재 본다.
지금은 "이 선택이 나중에 검색 성능을 좌우한다"만 알면 된다.

> **더 알아보기**: 세 번째 방법도 있다 — 표를 HTML 로 두고 `rowspan` 을 살리는 것.
> Docling 이 이 방식이다. 다만 의존성이 크고 CPU 에서 느려서 이 수업에서는 쓰지 않는다.

In [ ]:
# [셀 11] 표가 많은 문서 6건의 None 비율
# 표가 많은 문서 6건을 한 번에 재 본다
TABLE_DOCS = ["im_mortgage_2024", "kbstar_deposit_2021", "kakao_deposit_2026_pdf",
              "im_household_2025", "hanacard_diff_2016", "sf_realestate_loan_2026"]

print(f"{'doc_id':30}{'쪽':>4}{'표':>5}{'None셀':>8}{'None비율':>10}")
for did in TABLE_DOCS:
    p = path_of(did)
    if not p.exists():
        print(f"{did:30}  (없음)"); continue
    n_t = n_none = n_cell = 0
    with pdfplumber.open(p) as pdf:
        n_pages = len(pdf.pages)
        for page in pdf.pages:
            for tb in page.extract_tables():
                n_t += 1
                for row in tb:
                    n_cell += len(row)
                    n_none += sum(1 for c in row if c is None)
    ratio = n_none / n_cell if n_cell else 0
    print(f"{did:30}{n_pages:>4}{n_t:>5}{n_none:>8}{ratio:>10.1%}")

`None` 비율이 문서마다 크게 다르다. **높은 문서일수록 합쳐진 칸이 많다** = 표 구조가 복잡하다.

여러분의 분류표(`data/quality_table.csv`)에서 "PDF표중심"으로 분류한 문서와
이 목록이 겹치는지 확인해 보세요. 1회차 판단이 맞았는지 여기서 드러난다.

## 3부 · OCR

**수업 중에는 OCR 을 돌리지 않는다.** 스캔본 8쪽을 OCR 하는 데 몇 분이 걸리고,
엔진이 OS 마다 달라서(macOS Vision / RapidOCR) 결과도 조금씩 다르다.
6명이 각자 돌리면 6가지 결과가 나온다.

대신 **강사가 미리 돌려 둔 결과(캐시)를 읽는 구조**만 확인한다. 이게 실무에서도 맞는 방식이다.
OCR 은 비싸고 느리고 돌릴 때마다 결과가 조금씩 달라서, 파이프라인 안에서 매번 돌리면 안 된다.

In [ ]:
# [셀 12] OCR 캐시 읽기
from finrag.parsing.ocr import load_ocr, cache_path

doc_id = "knia_3500_scan"
print("캐시 위치:", cache_path(doc_id).relative_to(ROOT))
print("있는가:", cache_path(doc_id).exists())

pages, warnings = load_ocr(doc_id, source=path_of(doc_id))
if warnings:
    for w in warnings:
        print("  경고:", w)
if pages:
    print(f"\nOCR 결과 {len(pages)}쪽")
    first = pages[min(pages)]
    print("1쪽 앞부분:", repr(first[:120]))
else:
    print("\n캐시가 없습니다. 디스코드의 ocr_cache.zip 을 저장소의 data 폴더 안에 풀면 됩니다.")
    print("풀고 나면 이 파일이 있어야 합니다:", cache_path(doc_id).relative_to(ROOT))

### 캐시가 낡으면 조용히 쓰지 않는다

`load_ocr` 은 원본 PDF 의 SHA-256(파일 지문)을 캐시에 적힌 것과 비교한다. 다르면 경고를 낸다.

> **낡은 OCR 결과로 색인을 만드는 것이 가장 찾기 어려운 오류다.**
> 검색은 잘 되는데 답이 미묘하게 틀리고, 원본을 열어 보면 맞게 적혀 있다.

이런 "조용한 불일치"를 막는 장치를 파이프라인 곳곳에 둔다. 3회차 인제스천 그래프에서
다시 나온다.

## 4부 · 기준값은 어디서 오나

오늘 `validate.py` 의 TODO 를 채운다. 그 함수는 네 가지 수치로 쪽을 판정한다.

```
공백 비율 · 깨진 문자 비율 · 한글 비율 · 단음절 토큰 비율
```

수치마다 "이 값보다 나쁘면 걸러라"는 기준값이 있다. **기준값을 감으로 정하지 않는다.**
코퍼스 전체를 재고 분포를 보고 정한다. 아래 셀은 PDF 전체를 한 번 훑는다(약 7초).

In [ ]:
# [셀 13] PDF 1,306쪽 전체 — 네 지표의 분포
import statistics as st

rows = []
for did, r in MANIFEST.items():
    if not (r["filename"] or "").endswith(".pdf"):
        continue
    p = path_of(did)
    if not p.exists():
        continue
    try:
        pages = _pymupdf(p)
    except Exception:
        continue
    for i, text in enumerate(pages, 1):
        if len(text) >= 50:                 # 빈 페이지는 분포에서 뺀다
            rows.append((did, i, score_page(text)))

print(f"측정: PDF {len({d for d, _, _ in rows})}건 · {len(rows)}쪽\n")
print(f"{'지표':26}{'중앙값':>9}{'최소':>9}{'최대':>9}")
for key in ("space_ratio", "broken_ratio", "single_char_token_ratio", "hangul_ratio"):
    v = [m[key] for _, _, m in rows]
    print(f"{key:26}{st.median(v):>9.4f}{min(v):>9.4f}{max(v):>9.4f}")

In [ ]:
# [셀 14] 기준값을 적용하면 몇 쪽이 걸리나
from collections import Counter

THRESHOLDS = [
    ("space_ratio",             0.02, "<", "공백 소실"),
    ("broken_ratio",            0.05, ">", "폰트 깨짐"),
    ("single_char_token_ratio", 0.55, ">", "자간 분리"),
    ("hangul_ratio",            0.10, "<", "한국어 아님"),
]

for key, thr, op, label in THRESHOLDS:
    hit = [(d, i) for d, i, m in rows if (m[key] < thr if op == "<" else m[key] > thr)]
    top = Counter(d for d, _ in hit).most_common(3)
    print(f"{label:10} {key} {op} {thr}")
    print(f"    걸리는 쪽 {len(hit):>4}쪽 ({len(hit)/len(rows):.1%})   {top}")

### 이 숫자를 읽는 법

**띄어쓰기가 사라진 쪽이 138쪽(10.6%)** 이나 걸린다. 대부분 실손 표준약관과 손보협회 표준약관이다.
PyMuPDF 로 뽑았기 때문이다 — pdfplumber 로 다시 뽑으면 대부분 살아난다.
**게이트가 "다시 뽑아라(reparse)"로 보내는 것이 정확히 이 경우다.**

**폰트 깨짐은 4쪽(0.3%)** 뿐이다. 드물지만 걸리면 심각하다.

**단음절 토큰은 0쪽이다.** 기준값 0.55 에 걸리는 페이지가 하나도 없다.
코퍼스 최대값이 0.39 다.

> **그러면 이 검사를 빼야 하나?**
> 빼지 않는다. 글자가 한 자씩 떨어진 문서(`금 융 소 비 자`)가 나타나면 그 페이지는 단음절 비율이
> 0.9 를 넘는다. 지금 코퍼스에 없을 뿐이고, 걸렸을 때 놓치면 치명적이다.
> **한 번도 안 걸리는 검사를 두는 비용은 0 이고, 없어서 놓치는 비용은 크다.**
>
> 다만 **"0쪽이 걸린다"는 사실 자체를 기록해 둔다.** 나중에 이 검사가 갑자기
> 많이 걸리기 시작하면 코퍼스가 바뀐 것이다.

### 연습 — 기준값을 움직여 본다

공백 비율의 기준값을 바꾸면 몇 쪽이 걸리는지 본다. **너무 느슨하면 놓치고,
너무 빡빡하면 멀쩡한 문서까지 다시 뽑는 줄에 선다.**

In [ ]:
# [셀 15] 공백 비율의 기준값을 움직여 보기
print(f"{'임계값':>8}{'걸리는 쪽':>10}{'비율':>8}   대표 문서")
for thr in (0.005, 0.01, 0.02, 0.04, 0.08):
    hit = [(d, i) for d, i, m in rows if m["space_ratio"] < thr]
    top = Counter(d for d, _ in hit).most_common(2)
    print(f"{thr:>8.3f}{len(hit):>10}{len(hit)/len(rows):>8.1%}   {[t[0] for t in top]}")

**0.02 를 고른 이유**: 0.01 로 낮추면 실제로 띄어쓰기가 사라진 페이지를 놓치고,
0.04 로 올리면 원래 공백이 적은 표 페이지까지 걸린다.

여기서 원칙이 하나 더 나온다.

> **표를 위해 PyMuPDF 로 바꿔 끼운 페이지는 공백 검사에서 빼 준다.**
> 표는 원래 공백이 적다. 우리가 일부러 바꿔 놓고 그 결과를 벌주면 안 된다.

`validate.py` 의 `skip_space` 인자가 그것이다.

### 연습 2 (직접)

`broken_ratio` 의 기준값을 0.01 로 낮추면 몇 쪽이 걸리는가? 그중 **진짜 문제**는 몇 쪽인가?
걸린 페이지를 실제로 열어서 확인해 보세요.

In [ ]:
# [셀 16] 연습 — 깨진 문자 기준값을 낮춰 보기
# TODO: broken_ratio 임계값을 낮춰 가며 걸리는 페이지를 확인한다
for thr in (0.05,):                       # ← 0.01, 0.005 를 추가해 보세요
    hit = [(d, i, m["broken_ratio"]) for d, i, m in rows if m["broken_ratio"] > thr]
    print(f"임계 {thr}: {len(hit)}쪽")
    for d, i, r in sorted(hit, key=lambda x: -x[2])[:5]:
        print(f"   {d:30} {i:>4}쪽  {r:.5f}")

---

## 오늘 직접 채울 것

이 노트북에서 잰 것이 그대로 코드가 된다. 채울 함수는 셋이고, 쉬운 순서다.

| 순서 | 채울 함수 | 파일 | 확인 |
|---|---|---|---|
| 실습 1 | `normalize_article()` — "제 31 조" → "제31조" | `src/finrag/parsing/metadata.py` | `make test` 의 normalize 테스트 7개 |
| 실습 2 | `judge_page()` — 네 가지 수치로 pass / reparse / ocr | `src/finrag/parsing/validate.py` | judge_page 테스트 6개 |
| 실습 3 | `split_articles()` — 조항 머리글에서 자른다 | `src/finrag/parsing/chunk.py` | split_articles 테스트 4개 |

기준값 상수 다섯 개는 `validate.py` 맨 위에 있다. 위 4부에서 본 분포가 그 값의 근거다.
`extract()` 의 파서 고르기와 표 처리는 이미 완성돼 있다 — 읽어 두면 되고 채우지 않는다.

순서와 규칙은 `docs/session2/lab.md` 에 있다.

## 5부 · 고정 길이 vs 조항 단위

같은 문서를 두 방식으로 잘라 본다. 고정 길이는 구현이 쉽지만 조항 경계를 모른다.
"제7조 제2항의 통지 기한"이 두 조각에 걸치면 어느 쪽을 인용해도 근거가 반쪽이다.

In [ ]:
# [셀 17] 고정 500자 vs 조항 단위
import re

doc = "kakao_deposit_terms_2024"
text = "\n".join(_pdfplumber(path_of(doc)))
HEAD = re.compile(r"제\s*\d+\s*조(?:\s*의\s*\d+)?\s*[\[(（]?[^\])）\n]{0,20}")   # 조항 머리글
heads = [(m.start(), m.group(0).strip()) for m in HEAD.finditer(text)]
spans = [(s, heads[i + 1][0] if i + 1 < len(heads) else len(text), h) for i, (s, h) in enumerate(heads)]
print(f"{doc}: {len(text):,}자, 조항 {len(spans)}개")

# (1) 고정 길이로 자르면
SIZE = 500
cuts = list(range(SIZE, len(text), SIZE))
cut_articles = [(h, s, e) for s, e, h in spans if any(s < c < e for c in cuts)]
print(f"\n고정 {SIZE}자: 조각 {len(cuts) + 1}개. 조항 {len(cut_articles)}개가 조각 경계에 걸려 반으로 잘린다")
for h, s, e in cut_articles[:3]:
    c = next(c for c in cuts if s < c < e)
    print(f"  {h:24} …{text[c - 22:c]!r} | {text[c:c + 22]!r}…")

# (2) 조항 단위로 자르면 경계는 문서가 이미 정해 두었다. 대신 길이가 제각각이다.
short = [(h, e - s) for s, e, h in spans if e - s < 40]
print(f"\n조항 단위: 조각 {len(spans)}개. 그중 40자 미만 {len(short)}개:")
for h, n in short[:5]:
    print(f"  {h:24} {n}자")
print("\n하한을 40자로 두면 위 조항이 사라진다. chunk.py 의 MIN_CHARS_ARTICLE 이 12 인 이유다.")
